# Skin Cancer Detection — ISIC 2024
## Notebook 4: Out-of-Fold Target Encoding

**Goal:** Encode categorical columns without data leakage.

**Why OOF:** Global encoding leaks the answer into
training data. OOF encoding calculates each fold's
encoding using OTHER folds only — never itself.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

data = pd.read_csv('/content/cleaned_train_metadata.csv')

In [2]:
# split the data
y = data['target']
X = data.drop(columns=['target'])

print(X.shape)
print(y.shape)
print('cancer cases:' , y.sum())
print("Cancer rate:", round(y.mean() * 100, 4), "%")

(401059, 43)
(401059,)
cancer cases: 393
Cancer rate: 0.098 %


In [4]:
# OOF Target Encoding for anatom_site_general
data['anatom_site_encoded'] = np.nan


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 1

for train_idx,validation_idx in skf.split(X,y):
  train_data =  data.iloc[train_idx]
  val_data = data.iloc[validation_idx]



  encoding_map = train_data.groupby('anatom_site_general')['target'].mean()


  data.loc[data.index[validation_idx],'anatom_site_encoded'] = val_data['anatom_site_general'].map(encoding_map)

  print(f"Fold {fold} encoded!")
  fold +=1


print("\nMissing values:", data['anatom_site_encoded'].isnull().sum())



Fold 1 encoded!
Fold 2 encoded!
Fold 3 encoded!
Fold 4 encoded!
Fold 5 encoded!

Missing values: 0


In [9]:
# ⚠️ SIMULATION: Why Global Encoding Fails

# Global encoding — WRONG WAY
global_encoding = data.groupby(
    'anatom_site_general')['target'].mean()

data['global_encoded'] = data['anatom_site_general'].map(
    global_encoding)

print("=== GLOBAL ENCODING (LEAKY) ===")
print(data[['anatom_site_general',
            'global_encoded']].drop_duplicates())

print("\n=== OOF ENCODING (CORRECT) ===")
print(data[['anatom_site_general',
            'anatom_site_encoded']].drop_duplicates())

print("\nGlobal encoding uses ALL rows to calculate")
print("rates — including the rows being predicted.")
print("This leaks the answer into training! ❌")
print("\n✅ OOF encoding calculates rates using OTHER")
print("folds only — never the row being predicted.")

=== GLOBAL ENCODING (LEAKY) ===
  anatom_site_general  global_encoded
0     lower extremity        0.000709
1           head/neck        0.006475
2     posterior torso        0.000807
3      anterior torso        0.000934
6     upper extremity        0.000808

=== OOF ENCODING (CORRECT) ===
    anatom_site_general  anatom_site_encoded
0       lower extremity             0.000691
1             head/neck             0.006193
2       posterior torso             0.000802
3        anterior torso             0.000909
4        anterior torso             0.000941
6       upper extremity             0.000780
7       posterior torso             0.000756
8        anterior torso             0.000939
9       upper extremity             0.000743
10      lower extremity             0.000692
11      posterior torso             0.000832
13      upper extremity             0.000761
14      posterior torso             0.000823
15       anterior torso             0.000981
16      upper extremity          

## ⚠️ Why Global Target Encoding Causes Leakage

### Global Encoding (WRONG):
All 401,059 patients → calculate cancer rate
→ apply to SAME 401,059 patients
→ model sees its own answer during training ❌

### Out-of-Fold Encoding (CORRECT):
Fold 1 patients → encoded using Folds 2,3,4,5 ✅
Fold 2 patients → encoded using Folds 1,3,4,5 ✅
Fold 3 patients → encoded using Folds 1,2,4,5 ✅
Fold 4 patients → encoded using Folds 1,2,3,5 ✅
Fold 5 patients → encoded using Folds 1,2,3,4 ✅

No patient ever sees its own answer! ✅

### Key Insight:
Global encoding → inflated validation scores →
model fails on real patients
OOF encoding → honest validation scores →
model works on real patients